# 📊 Audit Sémantique des Lois de Finances du Cameroun — 2024 vs 2025
## Analyse statistique complète : glissement sémantique, évolution budgétaire & thématique

> **Contexte :** Ce notebook présente l'analyse comparative des Lois de Finances du Cameroun (2024–2025) 
> selon une approche NLP/Transformers combinée à des tests statistiques rigoureux.  
> **Objectifs :** Mesurer le glissement sémantique inter-annuel, tester la significativité des changements 
> thématiques, et évaluer l'alignement budgétaire avec la SND30.

---
**Piliers SND30 analysés :**
- 🔶 **Transformation structurelle** de l'économie pour accélérer la croissance économique  
- 🟢 **Capital humain** — Développement du capital humain et du bien-être social  
- 🟣 **Emploi et insertion** — Promotion de l'emploi et de l'insertion socio-économique  
- 🔵 **Gouvernance** — Décentralisation et gestion stratégique de l'État

---

## 0. Configuration & Chargement des données

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats
from scipy.stats import (
    chi2_contingency, mannwhitneyu, ks_2samp,
    shapiro, ttest_ind, kruskal, spearmanr
)
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
import itertools, re, os, textwrap
from collections import Counter

# ─── Styles ────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
matplotlib.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
})

# ─── Constantes métier ─────────────────────────────────────────────────────────
PILIERS_LONG = [
    "Transformation structurelle de l'économie pour accélérer la croissance economique",
    'Développement du capital humain et du bien-être social',
    "Promotion de l'emploi et de l'insertion socio-économique",
    "Gouvernance, décentralisation et gestion stratégique de l'État",
]
PILIERS_COURT = [
    'Transformation structurelle',
    'Capital humain',
    'Emploi et insertion',
    'Gouvernance',
]
PILIER_MAP = dict(zip(PILIERS_LONG, PILIERS_COURT))
PILIER_COLORS = {
    'Transformation structurelle': '#f97316',
    'Capital humain':              '#22c55e',
    'Emploi et insertion':         '#a855f7',
    'Gouvernance':                 '#0ea5e9',
}
YEAR_COLORS = {'2024': '#3b82f6', '2025': '#ef4444', 2024: '#3b82f6', 2025: '#ef4444'}
SEUIL = 0.70

def map_pilier(val):
    return PILIER_MAP.get(str(val), str(val))

def pct_fmt(v): return f'{v:.1f}%'
def mds_fmt(v): return f'{v/1e9:.2f} Mds'
def fmt_stat(v, n=4): return f'{v:.{n}f}' if not np.isnan(v) else 'N/A'
def stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '** '
    if p < 0.05:  return '*  '
    return 'n.s.'

print('✅ Imports et configuration OK')
print(f'  pandas {pd.__version__} | numpy {np.__version__}')

In [ ]:
# ─── Chargement des fichiers ───────────────────────────────────────────────────
DATA_PATH = '/mnt/user-data/uploads/'

# Budgets
budget_2024 = pd.read_excel(DATA_PATH + 'analyse_budgetaire.xlsx', sheet_name='Budget_2024')
budget_2025 = pd.read_excel(DATA_PATH + 'analyse_budgetaire.xlsx', sheet_name='Budget_2025')
piliers_2024 = pd.read_excel(DATA_PATH + 'analyse_budgetaire.xlsx', sheet_name='Piliers_2024')
piliers_2025 = pd.read_excel(DATA_PATH + 'analyse_budgetaire.xlsx', sheet_name='Piliers_2025')
comparaison  = pd.read_excel(DATA_PATH + 'analyse_budgetaire.xlsx', sheet_name='Comparaison_2024_2025')

# Normaliser les noms de piliers (colonnes courtes)
for df in [piliers_2024, piliers_2025, comparaison]:
    if 'pilier' in df.columns:
        df['pilier_court'] = df['pilier'].apply(map_pilier)

# Articles avec topics & clusters
art_top_24  = pd.read_excel(DATA_PATH + 'articles_2024_avec_topics.xlsx')
art_top_25  = pd.read_excel(DATA_PATH + 'articles_2025_avec_topics.xlsx')
art_cl_24   = pd.read_excel(DATA_PATH + 'articles_2024_avec_clusters.xlsx')
art_cl_25   = pd.read_excel(DATA_PATH + 'articles_2025_avec_clusters.xlsx')

# LDA topics
lda_24 = pd.read_excel(DATA_PATH + 'lda_topics_2024.xlsx')
lda_25 = pd.read_excel(DATA_PATH + 'lda_topics_2025.xlsx')

# Similarités embeddings
sim_all  = pd.read_excel(DATA_PATH + 'embeddings_similarities.xlsx')
sim_2024 = pd.read_excel(DATA_PATH + 'embeddings_similarities_2024.xlsx')
sim_2025 = pd.read_excel(DATA_PATH + 'embeddings_similarities_2025.xlsx')
sim_2024['annee'] = 2024
sim_2025['annee'] = 2025

# Objectifs classifiés SND30
obj_24 = pd.read_excel(DATA_PATH + 'objectifs_classifications_snd30_2024.xlsx')
obj_25 = pd.read_excel(DATA_PATH + 'objectifs_classifications_snd30_2025.xlsx')

# Mapping colonnes de score
SCORE_COLS = [c for c in obj_24.columns if c.startswith('score_') 
              and c not in ('score_pilier_dominant',)]
SCORE_LABELS = {c: map_pilier(c.replace('score_','')) for c in SCORE_COLS}

# Normaliser pilier_dominant
for df in [obj_24, obj_25]:
    df['pilier_court'] = df['pilier_dominant'].apply(map_pilier)

print('✅ Toutes les données chargées')
print(f'  Budget 2024 : {len(budget_2024)} lignes | Budget 2025 : {len(budget_2025)} lignes')
print(f'  Articles 2024 : {len(art_top_24)} | Articles 2025 : {len(art_top_25)}')
print(f'  Objectifs 2024 : {len(obj_24)} | Objectifs 2025 : {len(obj_25)}')
print(f'  Similarités totales : {len(sim_all)}')

---
## 1. Vue d'ensemble — Indicateurs clés & Portrait budgétaire
> Synthèse comparative des grandeurs macrobudgétaires 2024 vs 2025.

In [ ]:
# ─── KPIs macrobudgétaires ────────────────────────────────────────────────────
ae24 = budget_2024['ae'].sum(); ae25 = budget_2025['ae'].sum()
cp24 = budget_2024['cp'].sum(); cp25 = budget_2025['cp'].sum()

kpis = {
    'Total AE 2024':         ae24,
    'Total AE 2025':         ae25,
    'Évolution AE (%)':      (ae25-ae24)/ae24*100,
    'Total CP 2024':         cp24,
    'Total CP 2025':         cp25,
    'Évolution CP (%)':      (cp25-cp24)/cp24*100,
    'Ratio AE/CP 2024':      ae24/cp24,
    'Ratio AE/CP 2025':      ae25/cp25,
    'Nbre lignes 2024':      len(budget_2024),
    'Nbre lignes 2025':      len(budget_2025),
}

df_kpi = pd.DataFrame.from_dict(kpis, orient='index', columns=['Valeur'])
df_kpi['Formaté'] = df_kpi.apply(lambda r: (
    pct_fmt(r['Valeur']) if '%' in r.name or 'Ratio' in r.name
    else mds_fmt(r['Valeur']) if r['Valeur'] > 1e6
    else str(int(r['Valeur']))), axis=1)

print('='*55)
print(f'  PORTRAIT MACROBUDGÉTAIRE — LOI DE FINANCES CAMEROUN')
print('='*55)
for k, v in df_kpi.iterrows():
    print(f'  {k:<28}: {v["Formaté"]:>12}')
print('='*55)
print(f'  Variation AE  : {(ae25-ae24)/1e9:+.2f} Mds FCFA ({(ae25-ae24)/ae24*100:+.1f}%)')
print(f'  Variation CP  : {(cp25-cp24)/1e9:+.2f} Mds FCFA ({(cp25-cp24)/cp24*100:+.1f}%)')
print(f'  Écart AE-CP 2025 : {(ae25-cp25)/1e9:.2f} Mds FCFA')

In [ ]:
# ─── Figure 1 : Portrait budgétaire global ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Figure 1 — Portrait macrobudgétaire 2024 vs 2025', y=1.02)

# 1a. Barres AE et CP comparées
categories = ['AE', 'CP']
vals_24 = [ae24/1e9, cp24/1e9]
vals_25 = [ae25/1e9, cp25/1e9]
x = np.arange(len(categories))
w = 0.35
b1 = axes[0].bar(x - w/2, vals_24, w, label='2024', color='#3b82f6', alpha=0.85, zorder=3)
b2 = axes[0].bar(x + w/2, vals_25, w, label='2025', color='#ef4444', alpha=0.85, zorder=3)
axes[0].set_xticks(x); axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Milliards FCFA'); axes[0].set_title('AE et CP totaux (Mds FCFA)')
axes[0].legend()
for bar in list(b1)+list(b2):
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+0.05, f'{h:.2f}', ha='center', va='bottom', fontsize=9)
axes[0].grid(axis='y', alpha=0.4, zorder=0)

# 1b. Évolution par pilier (waterfall)
cmp = comparaison.copy()
cmp['pilier_court'] = cmp['pilier'].apply(map_pilier)
cmp['delta'] = (cmp['ae_2025'] - cmp['ae_2024']) / 1e9
bar_colors = ['#22c55e' if v >= 0 else '#ef4444' for v in cmp['delta']]
bars = axes[1].barh(cmp['pilier_court'], cmp['delta'], color=bar_colors, alpha=0.85, zorder=3)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Δ AE (Mds FCFA)'); axes[1].set_title('Variation AE par pilier (2025−2024)')
axes[1].set_yticklabels([textwrap.fill(l, 18) for l in cmp['pilier_court']], fontsize=9)
for bar in bars:
    w2 = bar.get_width()
    axes[1].text(w2 + (0.05 if w2>=0 else -0.05), bar.get_y()+bar.get_height()/2,
                 f'{w2:+.2f}', va='center', ha='left' if w2>=0 else 'right', fontsize=9)
axes[1].grid(axis='x', alpha=0.4, zorder=0)

# 1c. Donut AE 2025 par pilier
p25 = piliers_2025.copy()
p25['pilier_court'] = p25['pilier'].apply(map_pilier)
colors_pie = [PILIER_COLORS.get(pc, '#94a3b8') for pc in p25['pilier_court']]
wedges, texts, autotexts = axes[2].pie(
    p25['ae'], labels=None, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
    pctdistance=0.75
)
for at in autotexts: at.set_fontsize(9)
axes[2].set_title('Répartition AE 2025 par pilier SND30')
legend_patches = [mpatches.Patch(color=PILIER_COLORS.get(pc,'#94a3b8'), label=pc) for pc in p25['pilier_court']]
axes[2].legend(handles=legend_patches, loc='lower center', bbox_to_anchor=(0.5,-0.2), fontsize=8, ncol=1)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig1_portrait_budgetaire.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 1 sauvegardée')

In [ ]:
# ─── Tableau comparatif piliers ───────────────────────────────────────────────
print('\n📋 COMPARAISON PAR PILIER SND30')
print('='*90)
cmp2 = comparaison.copy()
cmp2['pilier_court'] = cmp2['pilier'].apply(map_pilier)
cmp2['AE 2024 (Mds)'] = cmp2['ae_2024'].apply(mds_fmt)
cmp2['CP 2024 (Mds)'] = cmp2['cp_2024'].apply(mds_fmt)
cmp2['AE 2025 (Mds)'] = cmp2['ae_2025'].apply(mds_fmt)
cmp2['CP 2025 (Mds)'] = cmp2['cp_2025'].apply(mds_fmt)
cmp2['Évol. AE (%)']  = cmp2['evolution'].apply(lambda x: f'{x:+.1f}%')

# Part relative
cmp2['Part AE 2024'] = (cmp2['ae_2024'] / cmp2['ae_2024'].sum() * 100).apply(pct_fmt)
cmp2['Part AE 2025'] = (cmp2['ae_2025'] / cmp2['ae_2025'].sum() * 100).apply(pct_fmt)

display_cols = ['pilier_court','AE 2024 (Mds)','Part AE 2024','AE 2025 (Mds)','Part AE 2025','Évol. AE (%)']
print(cmp2[display_cols].to_string(index=False))
print()
print('⚠️  Points saillants :')
max_evol = cmp2.loc[cmp2['evolution'].idxmax(), 'pilier_court']
min_evol = cmp2.loc[cmp2['evolution'].idxmin(), 'pilier_court']
print(f'  → Pilier en plus forte hausse  : {max_evol} (+{cmp2["evolution"].max():.1f}%)')
print(f'  → Pilier en recul              : {min_evol} ({cmp2["evolution"].min():+.1f}%)')

---
## 2. Analyse du Glissement Sémantique
> Mesure de la dérive lexicale entre les deux lois via la similarité cosinus des embeddings 
> (sentence-transformers/paraphrase-multilingual-mpnet-base-v2).

**Interprétation :** Un score proche de 1 = article quasiment inchangé. 
Sous le seuil **θ=0.70** → changement sémantique significatif.

In [ ]:
# ─── Statistiques descriptives des similarités ────────────────────────────────
sims_24 = sim_all[sim_all['annee']==2024]['score_max_similarite'].dropna()
sims_25 = sim_all[sim_all['annee']==2025]['score_max_similarite'].dropna()

def desc_sim(s, label):
    return {
        'Année': label,
        'N articles': len(s),
        'Moyenne': f'{s.mean():.4f}',
        'Médiane': f'{np.median(s):.4f}',
        'Écart-type': f'{s.std():.4f}',
        'Q1': f'{np.percentile(s,25):.4f}',
        'Q3': f'{np.percentile(s,75):.4f}',
        'Min': f'{s.min():.4f}',
        'Max': f'{s.max():.4f}',
        f'% < θ ({SEUIL})': f'{(s<SEUIL).mean()*100:.1f}%',
        'Glissement (1−moy)': f'{1-s.mean():.4f}',
    }

df_desc_sim = pd.DataFrame([desc_sim(sims_24, '2024'), desc_sim(sims_25, '2025')])
print('📊 STATISTIQUES DESCRIPTIVES — SIMILARITÉ COSINUS (EMBEDDINGS)')
print('='*65)
print(df_desc_sim.to_string(index=False))

# Score de glissement global
score_gliss_24 = 1 - sims_24.mean()
score_gliss_25 = 1 - sims_25.mean()
interp = lambda s: 'Faible' if s<0.2 else ('Modéré' if s<0.4 else 'Élevé')
print(f'\n🔴 Score de glissement 2024 : {score_gliss_24:.4f} → {interp(score_gliss_24)}')
print(f'🔴 Score de glissement 2025 : {score_gliss_25:.4f} → {interp(score_gliss_25)}')
print(f'   (articles 2025 ont en moyenne {sims_25.mean()-sims_24.mean():+.4f} de similarité vs 2024)')

In [ ]:
# ─── Tests statistiques sur la similarité ─────────────────────────────────────
print('\n🧪 TESTS STATISTIQUES — GLISSEMENT SÉMANTIQUE')
print('='*65)

# KS-test
ks_stat, ks_p = ks_2samp(sims_24, sims_25)
# Mann-Whitney U
mw_stat, mw_p = mannwhitneyu(sims_24, sims_25, alternative='two-sided')
# Cohen's d
def cohens_d(x, y):
    pooled = np.sqrt((x.std(ddof=1)**2 + y.std(ddof=1)**2)/2)
    return (x.mean()-y.mean())/pooled if pooled>0 else np.nan
def interpret_d(d):
    ad = abs(d)
    if np.isnan(d): return 'N/A'
    if ad<0.2: return 'négligeable'
    if ad<0.5: return 'faible'
    if ad<0.8: return 'moyen'
    return 'fort'

cd = cohens_d(sims_24.values, sims_25.values)
# Normalité (Shapiro si n≤5000)
_, p_norm_24 = shapiro(sims_24[:500])
_, p_norm_25 = shapiro(sims_25[:500])

print(f'  Kolmogorov-Smirnov  D={ks_stat:.4f}  p={ks_p:.4f}  {stars(ks_p)}')
print(f'  Mann-Whitney U      U={mw_stat:.0f}   p={mw_p:.4f}  {stars(mw_p)}')
print(f'  Cohen\'s d           d={cd:.4f}  → effet {interpret_d(cd)}')
print(f'  Shapiro-Wilk 2024   p={p_norm_24:.4f}  → {"Normal" if p_norm_24>0.05 else "Non-normal"}')
print(f'  Shapiro-Wilk 2025   p={p_norm_25:.4f}  → {"Normal" if p_norm_25>0.05 else "Non-normal"}')
print()
conclusion = 'significativement différentes' if ks_p < 0.05 else 'non significativement différentes'
print(f'  ✅ Conclusion : Les distributions de similarité 2024 et 2025 sont {conclusion}')
print(f'     → La loi 2025 présente une similarité cosinus PLUS ÉLEVÉE')
print(f'       (moindre glissement sémantique par rapport à 2024)')

In [ ]:
# ─── Figure 2 : Analyse du glissement sémantique ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Figure 2 — Analyse du glissement sémantique (similarité cosinus embeddings)')

# 2a. Histogramme densité comparé
ax = axes[0,0]
for sims, year, color in [(sims_24,'2024','#3b82f6'),(sims_25,'2025','#ef4444')]:
    ax.hist(sims, bins=30, density=True, alpha=0.5, color=color, label=f'{year}', zorder=3)
    # KDE
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(sims, bw_method=0.3)
    xk = np.linspace(sims.min()-0.05, sims.max()+0.05, 300)
    ax.plot(xk, kde(xk), color=color, lw=2.5)
ax.axvline(SEUIL, color='crimson', ls='--', lw=2, label=f'Seuil θ={SEUIL}')
ax.axvline(sims_24.mean(), color='#3b82f6', ls=':', lw=1.5, alpha=0.7)
ax.axvline(sims_25.mean(), color='#ef4444', ls=':', lw=1.5, alpha=0.7)
ax.set_xlabel('Similarité cosinus maximale'); ax.set_ylabel('Densité')
ax.set_title('Distribution des similarités (KDE)')
ax.legend(fontsize=9); ax.grid(alpha=0.3, zorder=0)

# 2b. ECDF
ax = axes[0,1]
for sims, year, color in [(sims_24,'2024','#3b82f6'),(sims_25,'2025','#ef4444')]:
    s_sorted = np.sort(sims)
    p = np.arange(1,len(s_sorted)+1)/len(s_sorted)
    ax.step(s_sorted, p, color=color, lw=2.5, label=year)
ax.axvline(SEUIL, color='crimson', ls='--', lw=2, label=f'θ={SEUIL}')
ax.set_xlabel('Similarité cosinus'); ax.set_ylabel('F(x) — Proportion cumulée')
ax.set_title('Fonctions de répartition empiriques (ECDF)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 2c. Violin + Box
ax = axes[0,2]
data_violin = [sims_24.values, sims_25.values]
vp = ax.violinplot(data_violin, positions=[1,2], showmedians=True, showextrema=True)
for i,(body,color) in enumerate(zip(vp['bodies'],['#3b82f6','#ef4444'])):
    body.set_facecolor(color); body.set_alpha(0.55)
ax.axhline(SEUIL, color='crimson', ls='--', lw=2, label=f'θ={SEUIL}')
ax.set_xticks([1,2]); ax.set_xticklabels(['2024','2025'])
ax.set_ylabel('Similarité cosinus'); ax.set_title('Violin + Médiane')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# 2d. Score de glissement par article (triés) — 2024
ax = axes[1,0]
sim24_sorted = np.sort(sims_24)
sim25_sorted = np.sort(sims_25[:len(sim24_sorted)] if len(sims_25)>=len(sim24_sorted) else sims_25)
ax.plot(sim24_sorted, color='#3b82f6', lw=1.5, label='2024', alpha=0.8)
ax.fill_between(range(len(sim24_sorted)), SEUIL, sim24_sorted,
                where=sim24_sorted<SEUIL, color='#ef4444', alpha=0.3, label=f'< θ={SEUIL}')
ax.axhline(SEUIL, color='crimson', ls='--', lw=1.5)
ax.set_xlabel('Articles (triés)'); ax.set_ylabel('Score similarité')
ax.set_title(f'Profil de similarité 2024 (n={len(sims_24)} articles)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 2e. Articles < seuil par chapitre (2024)
ax = axes[1,1]
merge_sim = sim_2024.merge(art_cl_24[['id','titre','chapitre']], on='id', how='left')
merge_sim['sous_seuil'] = merge_sim['score_max_similarite'] < SEUIL
if 'titre' in merge_sim.columns:
    grp = merge_sim.groupby('titre')['sous_seuil'].mean().sort_values(ascending=False).head(10)
    ypos = range(len(grp))
    colors_bar = ['#ef4444' if v>0.3 else '#f97316' if v>0.1 else '#22c55e' for v in grp.values]
    ax.barh(list(ypos), grp.values*100, color=colors_bar, alpha=0.85, zorder=3)
    ax.set_yticks(list(ypos))
    ax.set_yticklabels([textwrap.fill(t, 22) for t in grp.index], fontsize=8)
    ax.set_xlabel('% articles sous seuil'); ax.set_title('% articles < θ=0.70 par titre (2024)')
    ax.grid(axis='x', alpha=0.3, zorder=0)

# 2f. Scatterplot similarité vs rang article
ax = axes[1,2]
sim_sorted_all = sim_all.sort_values('score_max_similarite')
for annee, color in [(2024,'#3b82f6'),(2025,'#ef4444')]:
    sub = sim_sorted_all[sim_sorted_all['annee']==annee]
    ax.scatter(range(len(sub)), sub['score_max_similarite'].values,
               color=color, alpha=0.5, s=20, label=str(annee))
ax.axhline(SEUIL, color='crimson', ls='--', lw=2, label=f'θ={SEUIL}')
ax.set_xlabel('Article (rang)'); ax.set_ylabel('Similarité cosinus')
ax.set_title('Profil de glissement (tous articles)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig2_glissement_semantique.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 2 sauvegardée')

---
## 3. Analyse Topic Modeling (LDA)
> Modélisation thématique latente — 4 topics alignés sur les piliers SND30.
> On analyse les top mots, la prévalence, et le **glissement lexical** entre 2024 et 2025.

In [ ]:
# ─── Top mots par topic ────────────────────────────────────────────────────────
TOPIC_LABELS = {
    0: 'T0 – Fiscalité & revenus',
    1: 'T1 – Sanctions & sécurité sociale',
    2: 'T2 – Administration fiscale',
    3: 'T3 – Finances publiques & secteurs',
}

for year, lda in [(2024, lda_24), (2025, lda_25)]:
    print(f'\n📌 TOP MOTS PAR TOPIC — {year}')
    print('-'*60)
    for t in sorted(lda['topic'].unique()):
        sub = lda[lda['topic']==t].nlargest(8,'probability')
        words = ' | '.join([f'{row["word"]}({row["probability"]:.3f})' for _,row in sub.iterrows()])
        print(f'  {TOPIC_LABELS.get(t, f"Topic {t}"):30s}: {words}')

In [ ]:
# ─── Prévalence des topics (article dominant) ─────────────────────────────────
def count_topics(df, label):
    vc = df['dominant_topic'].value_counts().sort_index()
    vc.index = [TOPIC_LABELS.get(i,f'Topic {i}') for i in vc.index]
    vc_pct = (vc / vc.sum() * 100).round(1)
    return pd.DataFrame({'Nbre articles': vc, '% articles': vc_pct}).rename_axis('Topic')

prev_24 = count_topics(art_top_24, '2024')
prev_25 = count_topics(art_top_25, '2025')

print('\n📊 PRÉVALENCE DES TOPICS DOMINANTS')
print()
print('─── 2024 ───')
print(prev_24.to_string())
print('\n─── 2025 ───')
print(prev_25.to_string())

In [ ]:
# ─── Test Chi² sur la distribution des topics ──────────────────────────────────
combined_topics = pd.concat([
    art_top_24['dominant_topic'].to_frame().assign(annee='2024'),
    art_top_25['dominant_topic'].to_frame().assign(annee='2025'),
])
ct_topics = pd.crosstab(combined_topics['annee'], combined_topics['dominant_topic'])
chi2_t, p_t, dof_t, _ = chi2_contingency(ct_topics)

print('\n🧪 TEST CHI² — Distribution des topics dominants (2024 vs 2025)')
print('='*60)
print(f'  H₀ : même distribution de topics entre 2024 et 2025')
print(f'  χ² = {chi2_t:.4f}  |  ddl = {dof_t}  |  p = {p_t:.4f}  {stars(p_t)}')
conclusion_chi2 = 'REJETÉE' if p_t<0.05 else 'NON REJETÉE'
print(f'  → H₀ {conclusion_chi2} au seuil α=5%')
if p_t < 0.05:
    print('  → La distribution thématique a SIGNIFICATIVEMENT CHANGÉ entre 2024 et 2025')
else:
    print('  → La distribution thématique reste stable entre 2024 et 2025')

print()
print('📋 TABLE DE CONTINGENCE (nb articles par topic et année) :')
ct_norm = ct_topics.div(ct_topics.sum(axis=1), axis=0)*100
print((ct_topics.to_string() + '\n\n' + 'Proportions (%) :' + '\n' + ct_norm.round(1).to_string()))

In [ ]:
# ─── Figure 3 : Topic modeling ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Figure 3 — Analyse Topic Modeling LDA (4 topics × 2024–2025)')

# 3a. Prévalence comparative
ax = axes[0,0]
topics_sorted = sorted(art_top_24['dominant_topic'].unique())
t_labels = [TOPIC_LABELS.get(t, f'T{t}') for t in topics_sorted]
prev24_vals = [len(art_top_24[art_top_24['dominant_topic']==t]) for t in topics_sorted]
prev25_vals = [len(art_top_25[art_top_25['dominant_topic']==t]) for t in topics_sorted]
x = np.arange(len(topics_sorted))
ax.bar(x-0.2, prev24_vals, 0.35, label='2024', color='#3b82f6', alpha=0.85, zorder=3)
ax.bar(x+0.2, prev25_vals, 0.35, label='2025', color='#ef4444', alpha=0.85, zorder=3)
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(l,18) for l in t_labels], fontsize=8)
ax.set_ylabel('Nb articles'); ax.set_title('Prévalence des topics (topic dominant)')
ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)

# 3b. Top mots 2024 (top 2 topics)
for idx, (ax, year, lda) in enumerate([(axes[0,1],'2024',lda_24),(axes[0,2],'2025',lda_25)]):
    data_bars = []
    colors_bars = []
    palette = ['#3b82f6','#ef4444','#22c55e','#f97316']
    for t in sorted(lda['topic'].unique()):
        sub = lda[lda['topic']==t].nlargest(5,'probability')
        for _, row in sub.iterrows():
            data_bars.append({'word': row['word'], 'prob': row['probability'], 'topic': t})
            colors_bars.append(palette[t % 4])
    df_bars = pd.DataFrame(data_bars)
    ax.barh(range(len(df_bars)), df_bars['prob'], color=colors_bars, alpha=0.8, zorder=3)
    ax.set_yticks(range(len(df_bars)))
    ax.set_yticklabels(df_bars['word'], fontsize=7)
    ax.set_xlabel('Probabilité LDA'); ax.set_title(f'Top mots par topic — {year}')
    patches = [mpatches.Patch(color=palette[i%4], label=TOPIC_LABELS.get(i,f'T{i}')[:20]) 
               for i in sorted(lda['topic'].unique())]
    ax.legend(handles=patches, fontsize=7, loc='lower right')
    ax.grid(axis='x', alpha=0.3, zorder=0)

# 3c. Heatmap de contingence normalisée
ax = axes[1,0]
ct_norm2 = ct_topics.div(ct_topics.sum(axis=1), axis=0)*100
sns.heatmap(ct_norm2, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            xticklabels=[TOPIC_LABELS.get(c,f'T{c}')[:18] for c in ct_norm2.columns],
            cbar_kws={'label':'% (par année}'}, linewidths=0.5)
ax.set_title('Table contingence normalisée (%)')
ax.set_xlabel('Topic dominant'); ax.set_ylabel('Année')

# 3d. Glissement lexical : delta probabilité top mots
ax = axes[1,1]
all_words_24 = set(lda_24['word'].unique())
all_words_25 = set(lda_25['word'].unique())
d24_map = lda_24.groupby('word')['probability'].mean().to_dict()
d25_map = lda_25.groupby('word')['probability'].mean().to_dict()
all_words = all_words_24 | all_words_25
deltas = {w: d25_map.get(w,0) - d24_map.get(w,0) for w in all_words}
deltas_sorted = sorted(deltas.items(), key=lambda x: abs(x[1]), reverse=True)[:20]
words_d = [x[0] for x in deltas_sorted]
vals_d  = [x[1] for x in deltas_sorted]
colors_d = ['#22c55e' if v>0 else '#ef4444' for v in vals_d]
ax.barh(range(len(words_d)), vals_d, color=colors_d, alpha=0.85, zorder=3)
ax.set_yticks(range(len(words_d))); ax.set_yticklabels(words_d, fontsize=9)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Δ Probabilité (2025−2024)'); ax.set_title('Glissement lexical (top 20 mots)')
ax.grid(axis='x', alpha=0.3, zorder=0)

# 3e. Mots exclusifs
ax = axes[1,2]
only_24 = all_words_24 - all_words_25
only_25 = all_words_25 - all_words_24
shared  = all_words_24 & all_words_25
vals_set = [len(only_24), len(shared), len(only_25)]
labels_set = [f'Exclusif 2024\n(n={len(only_24)})', f'Commun\n(n={len(shared)})', f'Nouveau 2025\n(n={len(only_25)})']
bars_set = ax.bar(labels_set, vals_set, color=['#3b82f6','#94a3b8','#ef4444'], alpha=0.85, zorder=3)
for b in bars_set:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, str(int(b.get_height())),
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Nb mots'); ax.set_title('Vocabulaire LDA : exclusif vs commun')
ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig3_topic_modeling.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 3 sauvegardée')

---
## 4. Classification Zero-Shot SND30 — Objectifs budgétaires
> Classification des objectifs budgétaires selon les 4 piliers SND30 via 
> le modèle mDeBERTa-v3-base-mnli-xnli (zero-shot multilingue).
> On évalue la qualité des scores et on teste la significativité des glissements inter-piliers.

In [ ]:
# ─── Distribution des piliers classifiés ──────────────────────────────────────
print('📊 RÉPARTITION DES OBJECTIFS PAR PILIER DOMINANT')
print('='*75)
for year, obj in [(2024, obj_24), (2025, obj_25)]:
    vc = obj['pilier_court'].value_counts()
    total_ae = obj['ae'].sum()
    print(f'\n── {year} (n={len(obj)} objectifs) ──')
    print(f'  {'Pilier':<35} {'N':>5} {'%':>6}  {'AE (Mds)':>10} {'% AE':>6}')
    print('  ' + '─'*65)
    for pilier, cnt in vc.items():
        sub = obj[obj['pilier_court']==pilier]
        ae_p = sub['ae'].sum()
        print(f'  {pilier:<35} {cnt:>5} {cnt/len(obj)*100:>5.1f}%  {ae_p/1e9:>9.2f} {ae_p/total_ae*100:>5.1f}%')

In [ ]:
# ─── Tests statistiques : scores de classification ───────────────────────────
print('\n🧪 TESTS MANN-WHITNEY U — Scores de classification par pilier (2024 vs 2025)')
print('='*75)

results_mw = []
for col in SCORE_COLS:
    label = SCORE_LABELS[col]
    x24 = obj_24[col].dropna().values if col in obj_24.columns else np.array([])
    x25 = obj_25[col].dropna().values if col in obj_25.columns else np.array([])
    if len(x24)<2 or len(x25)<2: continue
    mw_stat, mw_p = mannwhitneyu(x24, x25, alternative='two-sided')
    d_val = cohens_d(x24, x25)
    delta_mean = x25.mean() - x24.mean()
    results_mw.append({
        'Pilier': label,
        'Moy.score 2024': f'{x24.mean():.4f}',
        'Moy.score 2025': f'{x25.mean():.4f}',
        'Δ moyen': f'{delta_mean:+.4f}',
        'U stat': f'{mw_stat:.0f}',
        'p-value': f'{mw_p:.4f}',
        'Sig.': stars(mw_p),
        "Cohen's d": f'{d_val:.3f}',
        'Effet': interpret_d(d_val),
        'Direction': '↑ 2025' if delta_mean>0 else '↓ 2025',
    })
    
df_mw = pd.DataFrame(results_mw)
print(df_mw.to_string(index=False))

# Chi² sur piliers dominants
ct_piliers = pd.crosstab(
    pd.Series(['2024']*len(obj_24) + ['2025']*len(obj_25), name='annee'),
    pd.concat([obj_24['pilier_court'], obj_25['pilier_court']], ignore_index=True).rename('pilier')
)
chi2_p, p_chi2_p, dof_p, _ = chi2_contingency(ct_piliers)
print(f'\n📌 Chi² piliers dominants : χ²={chi2_p:.4f}  ddl={dof_p}  p={p_chi2_p:.4f} {stars(p_chi2_p)}')
print(f'   → Distribution piliers {"SIGNIFICATIVEMENT" if p_chi2_p<0.05 else "NON significativement"} différente entre 2024 et 2025')

In [ ]:
# ─── Matrice de transition piliers ─────────────────────────────────────────────
# Jointure sur programme commun aux deux années
merge_prog = obj_24[['programme','pilier_court','ae']].rename(
    columns={'pilier_court':'pilier_2024','ae':'ae_2024'}).merge(
    obj_25[['programme','pilier_court','ae']].rename(
        columns={'pilier_court':'pilier_2025','ae':'ae_2025'}),
    on='programme', how='inner'
)
n_commun = len(merge_prog)
n_stable = (merge_prog['pilier_2024']==merge_prog['pilier_2025']).sum()
n_change = n_commun - n_stable
print(f'\n🔄 MATRICE DE TRANSITION DES PILIERS (programmes communs)')
print(f'   {n_commun} programmes en commun | {n_stable} stables ({n_stable/n_commun*100:.1f}%) | {n_change} reclassifiés ({n_change/n_commun*100:.1f}%)')
print()

transition = pd.crosstab(merge_prog['pilier_2024'], merge_prog['pilier_2025'],
                          rownames=['Pilier 2024 →'], colnames=['→ Pilier 2025'])
print(transition.to_string())
print()
# Identifier les flux les plus importants
for _, row in merge_prog[merge_prog['pilier_2024']!=merge_prog['pilier_2025']].head(8).iterrows():
    print(f'  {row["pilier_2024"]:28s} → {row["pilier_2025"]:28s}  ({row["programme"][:40]})')

In [ ]:
# ─── Figure 4 : Classification SND30 ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Figure 4 — Classification Zero-Shot SND30 des objectifs budgétaires')

# 4a. Barres nb objectifs par pilier
ax = axes[0,0]
p_labels = PILIERS_COURT
cnt24 = [len(obj_24[obj_24['pilier_court']==p]) for p in p_labels]
cnt25 = [len(obj_25[obj_25['pilier_court']==p]) for p in p_labels]
x = np.arange(len(p_labels))
ax.bar(x-0.2, cnt24, 0.35, label='2024', color='#3b82f6', alpha=0.85, zorder=3)
ax.bar(x+0.2, cnt25, 0.35, label='2025', color='#ef4444', alpha=0.85, zorder=3)
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(p,14) for p in p_labels], fontsize=8)
ax.set_ylabel('Nb objectifs'); ax.set_title('Objectifs par pilier dominant')
ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)

# 4b. AE par pilier
ax = axes[0,1]
ae24_p = [obj_24[obj_24['pilier_court']==p]['ae'].sum()/1e9 for p in p_labels]
ae25_p = [obj_25[obj_25['pilier_court']==p]['ae'].sum()/1e9 for p in p_labels]
ax.bar(x-0.2, ae24_p, 0.35, label='2024', color='#3b82f6', alpha=0.85, zorder=3)
ax.bar(x+0.2, ae25_p, 0.35, label='2025', color='#ef4444', alpha=0.85, zorder=3)
ax.set_xticks(x); ax.set_xticklabels([textwrap.fill(p,14) for p in p_labels], fontsize=8)
ax.set_ylabel('AE (Mds FCFA)'); ax.set_title('AE classifiées par pilier (Mds)')
ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)

# 4c. Scores de classification violin
ax = axes[0,2]
for i, col in enumerate(SCORE_COLS[:4]):
    lbl = SCORE_LABELS[col]
    color = list(PILIER_COLORS.values())[i]
    x24v = obj_24[col].dropna().values
    x25v = obj_25[col].dropna().values
    pos24 = i*3 + 0.7; pos25 = i*3 + 1.7
    vp24 = ax.violinplot([x24v], positions=[pos24], showmedians=True)
    vp25 = ax.violinplot([x25v], positions=[pos25], showmedians=True)
    for vp, c in [(vp24,'#3b82f6'),(vp25,'#ef4444')]:
        for b in vp.get('bodies',[]): b.set_facecolor(c); b.set_alpha(0.5)
ax.set_title('Distribution scores classification par pilier')
ax.set_ylabel('Score de probabilité')
ax.set_xticks([1.2, 4.2, 7.2, 10.2])
ax.set_xticklabels([textwrap.fill(p,12) for p in p_labels], fontsize=7.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#3b82f6',label='2024'),Patch(color='#ef4444',label='2025')],
          fontsize=9)
ax.grid(axis='y', alpha=0.3)

# 4d. Heatmap scores moyens piliers 2024 vs 2025
ax = axes[1,0]
score_matrix = np.array([[obj_24[c].mean(), obj_25[c].mean()] for c in SCORE_COLS])
sns.heatmap(score_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            xticklabels=['2024','2025'],
            yticklabels=[textwrap.fill(SCORE_LABELS[c],18) for c in SCORE_COLS],
            cbar_kws={'label':'Score moyen'}, linewidths=0.5)
ax.set_title('Scores moyens de classification par pilier')

# 4e. Matrice de transition heatmap
ax = axes[1,1]
if not transition.empty:
    sns.heatmap(transition, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[textwrap.fill(c,14) for c in transition.columns],
                yticklabels=[textwrap.fill(c,14) for c in transition.index],
                cbar_kws={'label':'Nb programmes'}, linewidths=0.5)
    ax.set_title(f'Matrice transition piliers\n({n_commun} programmes communs)')
    ax.set_xlabel('Pilier 2025'); ax.set_ylabel('Pilier 2024')
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=30)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

# 4f. Score pilier dominant — histogram
ax = axes[1,2]
ax.hist(obj_24['score_pilier_dominant'].dropna(), bins=25, alpha=0.55,
        color='#3b82f6', label='2024', density=True, zorder=3)
ax.hist(obj_25['score_pilier_dominant'].dropna(), bins=25, alpha=0.55,
        color='#ef4444', label='2025', density=True, zorder=3)
ax.set_xlabel('Score de confiance (pilier dominant)'); ax.set_ylabel('Densité')
ax.set_title('Confiance de classification (score pilier dominant)')
ax.legend(); ax.grid(alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig4_classification_snd30.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 4 sauvegardée')

---
## 5. Analyse du Clustering (K-Means & HDBSCAN)
> Analyse comparative des groupes d'articles identifiés par deux algorithmes complémentaires.
> Interprétation des clusters, stabilité inter-annuelle, et composition thématique.

In [ ]:
# ─── Statistiques des clusters ────────────────────────────────────────────────
print('📊 DISTRIBUTION DES CLUSTERS')
print('='*60)
for year, cl in [(2024, art_cl_24), (2025, art_cl_25)]:
    n_total = len(cl)
    print(f'\n── K-Means {year} (n={n_total}) ──')
    vc_km = cl['cluster_kmeans'].value_counts().sort_index()
    for c, cnt in vc_km.items():
        print(f'  Cluster {int(c):>1} : {cnt:>3} articles ({cnt/n_total*100:>5.1f}%)')
    
    if 'cluster_hdbscan' in cl.columns:
        print(f'\n── HDBSCAN {year} (n={n_total}) ──')
        vc_hdb = cl['cluster_hdbscan'].value_counts().sort_index()
        noise = int(vc_hdb.get(-1, 0))
        n_clusters = len([c for c in vc_hdb.index if c != -1])
        print(f'  Nb clusters (hors bruit) : {n_clusters}')
        print(f'  Points de bruit (−1)     : {noise} ({noise/n_total*100:.1f}%)')
        for c, cnt in vc_hdb.items():
            label = 'Bruit (−1)' if c==-1 else f'Cluster {int(c):>2}'
            print(f'  {label} : {cnt:>3} articles ({cnt/n_total*100:>5.1f}%)')

In [ ]:
# ─── Composition thématique des clusters ──────────────────────────────────────
# Fusionner clusters + topics
merged_24 = art_cl_24[['id','titre','cluster_kmeans','cluster_hdbscan']].merge(
    art_top_24[['id','dominant_topic']], on='id', how='left')
merged_25 = art_cl_25[['id','titre','cluster_kmeans','cluster_hdbscan']].merge(
    art_top_25[['id','dominant_topic']], on='id', how='left')

for year, df in [(2024, merged_24), (2025, merged_25)]:
    print(f'\n📌 COMPOSITION THÉMATIQUE DES CLUSTERS K-MEANS — {year}')
    print('-'*65)
    ct = pd.crosstab(df['cluster_kmeans'], df['dominant_topic'])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.columns = [TOPIC_LABELS.get(c,f'T{c}')[:25] for c in ct_pct.columns]
    print(ct_pct.round(1).to_string())

# Test Kruskal-Wallis : similarité cosinus diffère-t-elle entre clusters ?
print('\n\n🧪 TEST KRUSKAL-WALLIS — Similarité cosinus selon cluster K-Means (2024)')
m_sim_24 = sim_2024.merge(art_cl_24[['id','cluster_kmeans']], on='id', how='left').dropna()
groups_kw = [m_sim_24[m_sim_24['cluster_kmeans']==c]['score_max_similarite'].values
             for c in sorted(m_sim_24['cluster_kmeans'].unique())]
kw_stat, kw_p = kruskal(*groups_kw)
print(f'  H stat = {kw_stat:.4f}  |  p = {kw_p:.4f}  {stars(kw_p)}')
if kw_p < 0.05:
    print('  → Les clusters présentent des distributions de similarité SIGNIFICATIVEMENT différentes')
    # Post-hoc pairwise
    clusters = sorted(m_sim_24['cluster_kmeans'].unique())
    print('  Post-hoc Mann-Whitney (pairwise) :')
    for c1, c2 in itertools.combinations(clusters, 2):
        g1 = m_sim_24[m_sim_24['cluster_kmeans']==c1]['score_max_similarite'].values
        g2 = m_sim_24[m_sim_24['cluster_kmeans']==c2]['score_max_similarite'].values
        _, p12 = mannwhitneyu(g1, g2, alternative='two-sided')
        print(f'    Cluster {int(c1)} vs {int(c2)} : p = {p12:.4f}  {stars(p12)}')

In [ ]:
# ─── Figure 5 : Clustering ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Figure 5 — Analyse du Clustering K-Means & HDBSCAN')

# 5a. Taille clusters K-Means
ax = axes[0,0]
km_24 = art_cl_24['cluster_kmeans'].value_counts().sort_index()
km_25 = art_cl_25['cluster_kmeans'].value_counts().sort_index()
all_cl = sorted(set(km_24.index)|set(km_25.index))
x = np.arange(len(all_cl))
ax.bar(x-0.2, [km_24.get(c,0) for c in all_cl], 0.35, label='2024', color='#3b82f6', alpha=0.85, zorder=3)
ax.bar(x+0.2, [km_25.get(c,0) for c in all_cl], 0.35, label='2025', color='#ef4444', alpha=0.85, zorder=3)
ax.set_xticks(x); ax.set_xticklabels([f'Cluster {int(c)}' for c in all_cl])
ax.set_ylabel('Nb articles'); ax.set_title('Taille des clusters K-Means (3 clusters)')
ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)

# 5b. HDBSCAN bruit
ax = axes[0,1]
hdb_24 = art_cl_24['cluster_hdbscan'].value_counts().sort_index()
hdb_25 = art_cl_25['cluster_hdbscan'].value_counts().sort_index()
all_hdb = sorted(set(hdb_24.index)|set(hdb_25.index))
colors_hdb = ['#94a3b8' if c==-1 else '#3b82f6' for c in all_hdb]
ax.bar([f'{int(c)}' for c in all_hdb],
       [hdb_24.get(c,0) for c in all_hdb], color=colors_hdb, alpha=0.85, zorder=3, label='2024')
ax.set_ylabel('Nb articles (2024)'); ax.set_title('Clusters HDBSCAN 2024 (−1 = bruit)')
ax.legend(); ax.grid(axis='y', alpha=0.3, zorder=0)

# 5c. Boxplot similarité par cluster
ax = axes[0,2]
m_sim_all = pd.concat([
    sim_2024.merge(art_cl_24[['id','cluster_kmeans']], on='id', how='left'),
    sim_2025.merge(art_cl_25[['id','cluster_kmeans']], on='id', how='left'),
]).dropna(subset=['cluster_kmeans'])
palette = {'2024': '#3b82f6', '2025': '#ef4444'}
for cl_id in sorted(m_sim_all['cluster_kmeans'].unique()):
    sub = m_sim_all[m_sim_all['cluster_kmeans']==cl_id]['score_max_similarite']
    ax.boxplot(sub, positions=[cl_id], widths=0.4, patch_artist=True,
               boxprops=dict(facecolor='#3b82f660'),
               medianprops=dict(color='#1e3a5f', linewidth=2))
ax.axhline(SEUIL, color='crimson', ls='--', lw=2, label=f'θ={SEUIL}')
ax.set_xlabel('Cluster K-Means'); ax.set_ylabel('Similarité cosinus')
ax.set_title('Similarité cosinus par cluster (2024+2025)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# 5d. Composition thématique empilée K-Means 2024
ax = axes[1,0]
ct24 = pd.crosstab(merged_24['cluster_kmeans'], merged_24['dominant_topic'])
ct24_pct = ct24.div(ct24.sum(axis=1),axis=0)*100
bottom = np.zeros(len(ct24_pct))
colors_stk = ['#3b82f6','#ef4444','#22c55e','#f97316']
for i, col in enumerate(ct24_pct.columns):
    lbl = TOPIC_LABELS.get(col,f'T{col}')[:20]
    ax.bar(ct24_pct.index, ct24_pct[col], bottom=bottom,
           color=colors_stk[i%4], alpha=0.85, label=lbl, zorder=3)
    bottom += ct24_pct[col].values
ax.set_xticks(ct24_pct.index); ax.set_xticklabels([f'Cluster {int(c)}' for c in ct24_pct.index])
ax.set_ylabel('%'); ax.set_title('Composition thématique clusters 2024')
ax.legend(fontsize=7, loc='upper right'); ax.grid(axis='y', alpha=0.3, zorder=0)

# 5e. Même pour 2025
ax = axes[1,1]
ct25 = pd.crosstab(merged_25['cluster_kmeans'], merged_25['dominant_topic'])
ct25_pct = ct25.div(ct25.sum(axis=1),axis=0)*100
bottom = np.zeros(len(ct25_pct))
for i, col in enumerate(ct25_pct.columns):
    lbl = TOPIC_LABELS.get(col,f'T{col}')[:20]
    ax.bar(ct25_pct.index, ct25_pct[col], bottom=bottom,
           color=colors_stk[i%4], alpha=0.85, label=lbl, zorder=3)
    bottom += ct25_pct[col].values
ax.set_xticks(ct25_pct.index); ax.set_xticklabels([f'Cluster {int(c)}' for c in ct25_pct.index])
ax.set_ylabel('%'); ax.set_title('Composition thématique clusters 2025')
ax.legend(fontsize=7, loc='upper right'); ax.grid(axis='y', alpha=0.3, zorder=0)

# 5f. Heatmap cross-clusters 2024 vs 2025 (via programme commun)
ax = axes[1,2]
cl_cross = art_cl_24[['id','cluster_kmeans']].rename(columns={'cluster_kmeans':'cl_24'}).merge(
    art_cl_25[['id','cluster_kmeans']].rename(columns={'cluster_kmeans':'cl_25'}), on='id', how='inner')
if not cl_cross.empty:
    ct_cl = pd.crosstab(cl_cross['cl_24'], cl_cross['cl_25'])
    sns.heatmap(ct_cl, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[f'Cl.{int(c)}-25' for c in ct_cl.columns],
                yticklabels=[f'Cl.{int(c)}-24' for c in ct_cl.index],
                linewidths=0.5)
    ax.set_title('Cross-clusters articles communs (2024→2025)')
    ax.set_xlabel('Cluster 2025'); ax.set_ylabel('Cluster 2024')
else:
    ax.text(0.5,0.5,'Pas d\'articles communs\n(IDs différents)', ha='center',va='center',
            transform=ax.transAxes, fontsize=11)
    ax.set_title('Cross-clusters articles communs')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig5_clustering.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 5 sauvegardée')

---
## 6. Tableau de Bord Statistique Consolidé
> Récapitulatif de tous les tests et indicateurs calculés dans ce notebook.
> Concentration budgétaire, courbes de Lorenz, Gini, HHI et synthèse des tests.

In [ ]:
# ─── Indice de Gini et HHI ────────────────────────────────────────────────────
def gini(x):
    x = np.sort(np.abs(x[np.isfinite(x)]))
    n = len(x)
    if n == 0 or x.sum() == 0: return float('nan')
    return (2*np.sum((np.arange(1,n+1))*x)/(n*x.sum()) - (n+1)/n)

def hhi(shares):
    s = np.abs(shares); s = s/s.sum()
    return float((s**2).sum())

g24_ae = gini(budget_2024['ae'].values)
g25_ae = gini(budget_2025['ae'].values)
g24_cp = gini(budget_2024['cp'].values)
g25_cp = gini(budget_2025['cp'].values)
hhi_ae_pilier_24 = hhi(piliers_2024['ae'].values)
hhi_ae_pilier_25 = hhi(piliers_2025['ae'].values)

print('📐 INDICES DE CONCENTRATION BUDGÉTAIRE')
print('='*55)
print(f'  Gini AE 2024       : {g24_ae:.4f}')
print(f'  Gini AE 2025       : {g25_ae:.4f}  (Δ = {g25_ae-g24_ae:+.4f})')
print(f'  Gini CP 2024       : {g24_cp:.4f}')
print(f'  Gini CP 2025       : {g25_cp:.4f}  (Δ = {g25_cp-g24_cp:+.4f})')
print(f'  HHI piliers AE 2024: {hhi_ae_pilier_24*10000:.1f} /10000')
print(f'  HHI piliers AE 2025: {hhi_ae_pilier_25*10000:.1f} /10000  (Δ = {(hhi_ae_pilier_25-hhi_ae_pilier_24)*10000:+.1f})')
print()
print('  Référence HHI : < 1500 = peu concentré | 1500–2500 = modéré | > 2500 = très concentré')
conc = 'peu concentré' if hhi_ae_pilier_25*10000 < 1500 else ('modéré' if hhi_ae_pilier_25*10000 < 2500 else 'très concentré')
print(f'  → Budget 2025 {conc} ({hhi_ae_pilier_25*10000:.0f})')

In [ ]:
# ─── Courbes de Lorenz ────────────────────────────────────────────────────────
def lorenz_curve(x):
    s = np.sort(np.abs(x[np.isfinite(x)]))
    cum = np.cumsum(s)/s.sum()
    return np.concatenate([[0], cum])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Figure 6 — Concentration budgétaire & Courbes de Lorenz')

# 6a. Lorenz AE
ax = axes[0]
for year, b, color in [(2024,budget_2024,'#3b82f6'),(2025,budget_2025,'#ef4444')]:
    lc = lorenz_curve(b['ae'].values)
    x_lc = np.linspace(0, 1, len(lc))
    g = gini(b['ae'].values)
    ax.plot(x_lc, lc, color=color, lw=2.5, label=f'{year} (Gini={g:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1.5, alpha=0.5, label='Égalité parfaite')
ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('Fraction cumulée des lignes'); ax.set_ylabel('Fraction cumulée des AE')
ax.set_title('Courbes de Lorenz — AE'); ax.legend(); ax.grid(alpha=0.3)

# 6b. Top 10% programmes
ax = axes[1]
for year, b, color in [(2024,budget_2024,'#3b82f6'),(2025,budget_2025,'#ef4444')]:
    s = np.sort(b['ae'].values)[::-1]
    top_pcts = [s[:max(1,int(len(s)*p/100))].sum()/s.sum()*100 for p in [5,10,20,30,50]]
    ax.plot([5,10,20,30,50], top_pcts, 'o-', color=color, lw=2.5, markersize=7, label=str(year))
ax.set_xlabel('% des lignes (triées par AE décroissant)')
ax.set_ylabel('% des AE cumulées')
ax.set_title('Courbe de concentration (top-N programmes)')
ax.legend(); ax.grid(alpha=0.3)
# Annoter
for p, v in zip([5,10,20,30,50], [s[:max(1,int(len(s)*p/100))].sum()/s.sum()*100 
                                   for p in [5,10,20,30,50]]):
    ax.annotate(f'{v:.0f}%', (p,v), textcoords='offset points', xytext=(5,3), fontsize=8)

# 6c. Pareto AE 2025
ax = axes[2]
b25_sorted = budget_2025.sort_values('ae', ascending=False).reset_index(drop=True)
b25_sorted['pct_cum_ae'] = b25_sorted['ae'].cumsum()/b25_sorted['ae'].sum()*100
b25_sorted['rang'] = range(1, len(b25_sorted)+1)
ax.bar(b25_sorted['rang'], b25_sorted['ae']/b25_sorted['ae'].sum()*100,
       color='#3b82f6', alpha=0.7, zorder=3)
ax2 = ax.twinx()
ax2.plot(b25_sorted['rang'], b25_sorted['pct_cum_ae'], 'r-', lw=2.5, label='% AE cumulées')
ax2.axhline(80, color='crimson', ls='--', lw=1.5)
ax2.set_ylabel('% AE cumulées', color='red')
# Trouver ligne 80%
ligne_80 = b25_sorted[b25_sorted['pct_cum_ae']>=80].index[0]+1
ax2.axvline(ligne_80, color='crimson', ls=':', lw=1.5)
ax2.annotate(f'{ligne_80} lignes = 80% AE', (ligne_80, 80),
             textcoords='offset points', xytext=(5,-15), fontsize=9, color='crimson')
ax.set_xlabel('Rang des programmes (AE décroissant)')
ax.set_ylabel('% AE individuel')
ax.set_title('Diagramme de Pareto AE 2025')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax2.legend(loc='center right')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig6_concentration.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Figure 6 sauvegardée | Règle de Pareto : {ligne_80} lignes représentent 80% des AE en 2025')

In [ ]:
# ─── Figure 7 : Tableau de bord statistique final ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Figure 7 — Tableau de Bord Statistique Consolidé', fontsize=16)

# 7a. Résumé des tests (heatmap p-values)
ax = axes[0,0]
tests_summary = {
    'KS (similarité cosinus)':    ks_p,
    'Mann-Whitney (similarité)':  mw_p,
    'Chi² (topics dominants)':    p_t,
    'Chi² (piliers SND30)':       p_chi2_p,
    'Kruskal (clusters vs sim.)': kw_p,
}
test_names = list(tests_summary.keys())
p_vals     = list(tests_summary.values())
colors_test = ['#22c55e' if p<0.05 else '#ef4444' for p in p_vals]
bars_t = ax.barh(range(len(test_names)), [-np.log10(max(p,1e-10)) for p in p_vals],
                 color=colors_test, alpha=0.85, zorder=3)
ax.axvline(-np.log10(0.05), color='crimson', ls='--', lw=2, label='α=0.05')
ax.set_yticks(range(len(test_names)))
ax.set_yticklabels([textwrap.fill(t,28) for t in test_names], fontsize=8.5)
ax.set_xlabel('−log₁₀(p)')
ax.set_title('Tests statistiques — volcano view')
ax.legend(fontsize=9)
for i, (b, p) in enumerate(zip(bars_t, p_vals)):
    w = b.get_width()
    ax.text(w+0.05, b.get_y()+b.get_height()/2,
            f'p={p:.3f} {stars(p)}', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3, zorder=0)

# 7b. Cohen's d par pilier
ax = axes[0,1]
d_vals_pilier = []
p_labels_plot = []
for col in SCORE_COLS:
    x24 = obj_24[col].dropna().values if col in obj_24.columns else np.array([])
    x25 = obj_25[col].dropna().values if col in obj_25.columns else np.array([])
    if len(x24)>1 and len(x25)>1:
        d_vals_pilier.append(cohens_d(x24,x25))
        p_labels_plot.append(SCORE_LABELS[col])
colors_cd = ['#22c55e' if v>0 else '#ef4444' for v in d_vals_pilier]
ax.barh(range(len(p_labels_plot)), d_vals_pilier, color=colors_cd, alpha=0.85, zorder=3)
for thresh, label in [(0.2,'faible'),(0.5,'moyen'),(0.8,'fort')]:
    ax.axvline(thresh, color='#94a3b8', ls=':', lw=1.5)
    ax.axvline(-thresh, color='#94a3b8', ls=':', lw=1.5)
ax.axvline(0, color='black', lw=1.5)
ax.set_yticks(range(len(p_labels_plot)))
ax.set_yticklabels([textwrap.fill(p,20) for p in p_labels_plot], fontsize=8.5)
ax.set_xlabel("Cohen's d")
ax.set_title("Taille d'effet (Cohen's d) — scores piliers")
for i, d in enumerate(d_vals_pilier):
    ax.text(d+(0.02 if d>=0 else -0.02), i, f'{d:.3f} ({interpret_d(d)})',
            va='center', ha='left' if d>=0 else 'right', fontsize=8)
ax.grid(axis='x', alpha=0.3, zorder=0)

# 7c. Évolution comparative AE vs CP vs similarité (radar normalisé)
ax = axes[0,2]
categories_radar = ['AE', 'CP', 'Nb lignes', 'Sim. moy.\n2024', 'Sim. moy.\n2025']
vals_radar = [
    ae25/ae24, cp25/cp24, len(budget_2025)/len(budget_2024),
    sims_25.mean()/sims_24.mean(), 1.0
]
bars_r = ax.bar(categories_radar, [(v-1)*100 for v in vals_radar],
                color=['#22c55e' if v>0 else '#ef4444' for v in [(v-1)*100 for v in vals_radar]],
                alpha=0.85, zorder=3)
ax.axhline(0, color='black', lw=1.5)
ax.set_ylabel('Variation (%)'); ax.set_title('Variations relatives 2025/2024 (%)')
for b in bars_r:
    h = b.get_height()
    ax.text(b.get_x()+b.get_width()/2, h+(0.3 if h>=0 else -0.8),
            f'{h:+.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3, zorder=0)

# 7d. Spearman AE vs score pilier dominant
ax = axes[1,0]
for year, obj, color in [(2024,obj_24,'#3b82f6'),(2025,obj_25,'#ef4444')]:
    valid = obj[['ae','score_pilier_dominant']].dropna()
    if len(valid) > 5:
        rho, p_sp = spearmanr(valid['ae'], valid['score_pilier_dominant'])
        ax.scatter(valid['ae']/1e6, valid['score_pilier_dominant'],
                   color=color, alpha=0.4, s=25, label=f'{year} (ρ={rho:.3f}, p={p_sp:.3f})')
ax.set_xlabel('AE (Millions FCFA)'); ax.set_ylabel('Score de confiance (pilier dominant)')
ax.set_title('Corrélation Spearman : AE vs confiance SND30')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xscale('log')

# 7e. Distribution score de confiance par pilier 2024 vs 2025
ax = axes[1,1]
for i, pilier in enumerate(PILIERS_COURT[:4]):
    pos = i*2.5
    d24v = obj_24[obj_24['pilier_court']==pilier]['score_pilier_dominant'].dropna().values
    d25v = obj_25[obj_25['pilier_court']==pilier]['score_pilier_dominant'].dropna().values
    if len(d24v)>0: ax.boxplot([d24v], positions=[pos], widths=0.8, patch_artist=True,
                                boxprops=dict(facecolor='#3b82f660'),
                                medianprops=dict(color='#1e3a5f', lw=2), zorder=3)
    if len(d25v)>0: ax.boxplot([d25v], positions=[pos+1], widths=0.8, patch_artist=True,
                                boxprops=dict(facecolor='#ef444460'),
                                medianprops=dict(color='darkred', lw=2), zorder=3)
ax.set_xticks([0.5, 3.0, 5.5, 8.0])
ax.set_xticklabels([textwrap.fill(p,14) for p in PILIERS_COURT[:4]], fontsize=8)
ax.set_ylabel('Score de confiance (pilier dominant)')
ax.set_title('Score de confiance par pilier (bleu=2024, rouge=2025)')
ax.legend(handles=[Patch(color='#3b82f660',label='2024'),Patch(color='#ef444460',label='2025')],fontsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)

# 7f. Synthèse : nb indicateurs significatifs
ax = axes[1,2]
all_tests = [
    ('KS similarité', ks_p),
    ('MW similarité', mw_p),
    ('Chi² topics', p_t),
    ('Chi² piliers', p_chi2_p),
    ('Kruskal clusters', kw_p),
]
mw_pilier_ps = []
for col in SCORE_COLS:
    x24 = obj_24[col].dropna().values if col in obj_24.columns else np.array([])
    x25 = obj_25[col].dropna().values if col in obj_25.columns else np.array([])
    if len(x24)>1 and len(x25)>1:
        _, pp = mannwhitneyu(x24, x25, alternative='two-sided')
        all_tests.append((f'MW {SCORE_LABELS[col]}', pp))

sig_001 = sum(1 for _,p in all_tests if p<0.001)
sig_01  = sum(1 for _,p in all_tests if 0.001<=p<0.01)
sig_05  = sum(1 for _,p in all_tests if 0.01<=p<0.05)
non_sig = sum(1 for _,p in all_tests if p>=0.05)

labels_pie2 = [f'p<0.001 (***) n={sig_001}',f'p<0.01 (**) n={sig_01}',
               f'p<0.05 (*) n={sig_05}',f'n.s. n={non_sig}']
sizes_pie2 = [sig_001, sig_01, sig_05, non_sig]
colors_pie2 = ['#ef4444','#f97316','#eab308','#94a3b8']
if sum(sizes_pie2) > 0:
    ax.pie([s for s in sizes_pie2 if s>0],
           labels=[l for l,s in zip(labels_pie2,sizes_pie2) if s>0],
           colors=[c for c,s in zip(colors_pie2,sizes_pie2) if s>0],
           autopct='%1.0f%%', startangle=90,
           wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title(f'Bilan des tests ({len(all_tests)} tests au total)')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/fig7_tableau_bord_stats.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Figure 7 sauvegardée')

---
## 7. Synthèse des Résultats & Recommandations

### 7.1 Récapitulatif de l'audit sémantique

In [ ]:
# ─── Tableau de synthèse final ────────────────────────────────────────────────
print('='*80)
print('  RAPPORT DE SYNTHÈSE — AUDIT SÉMANTIQUE LOI DE FINANCES CAMEROUN')
print('  2024 vs 2025 | ISE3-DS, ISSEA Yaoundé')
print('='*80)

print('\n📌 1. DIMENSION BUDGÉTAIRE')
print(f'   AE 2024  : {ae24/1e9:.2f} Mds FCFA | AE 2025 : {ae25/1e9:.2f} Mds ({(ae25-ae24)/ae24*100:+.1f}%)')
print(f'   CP 2024  : {cp24/1e9:.2f} Mds FCFA | CP 2025 : {cp25/1e9:.2f} Mds ({(cp25-cp24)/cp24*100:+.1f}%)')
print(f'   Gini AE : {g24_ae:.3f} (2024) → {g25_ae:.3f} (2025)  |  HHI piliers : {hhi_ae_pilier_24*10000:.0f} → {hhi_ae_pilier_25*10000:.0f}')

print('\n   Par pilier SND30 :')
for _, row in comparaison.iterrows():
    symbol = '↑' if row['evolution']>0 else '↓'
    print(f'   {symbol} {map_pilier(row["pilier"]):30s}: AE {row["ae_2024"]/1e9:.2f} → {row["ae_2025"]/1e9:.2f} Mds ({row["evolution"]:+.1f}%)')

print('\n📌 2. GLISSEMENT SÉMANTIQUE (EMBEDDINGS)')
print(f'   Similarité cosinus moy. 2024 : {sims_24.mean():.4f} | Score glissement : {1-sims_24.mean():.4f}')
print(f'   Similarité cosinus moy. 2025 : {sims_25.mean():.4f} | Score glissement : {1-sims_25.mean():.4f}')
print(f'   Articles 2024 sous seuil θ=0.70 : {(sims_24<SEUIL).sum()} ({(sims_24<SEUIL).mean()*100:.1f}%)')
print(f'   Articles 2025 sous seuil θ=0.70 : {(sims_25<SEUIL).sum()} ({(sims_25<SEUIL).mean()*100:.1f}%)')
print(f'   KS-test : D={ks_stat:.4f}, p={ks_p:.4f} {stars(ks_p)} | Cohen\'s d = {cd:.3f} ({interpret_d(cd)})')

print('\n📌 3. TOPIC MODELING LDA')
for t, label in TOPIC_LABELS.items():
    n24 = len(art_top_24[art_top_24['dominant_topic']==t])
    n25 = len(art_top_25[art_top_25['dominant_topic']==t])
    if n24+n25 > 0:
        print(f'   {label[:45]:45s}: 2024={n24:>3} arts → 2025={n25:>3} arts ({n25-n24:+d})')
print(f'   Chi² distribution : χ²={chi2_t:.2f}, p={p_t:.4f} {stars(p_t)}')

print('\n📌 4. CLASSIFICATION SND30 — OBJECTIFS BUDGÉTAIRES')
for pilier in PILIERS_COURT:
    n24 = len(obj_24[obj_24['pilier_court']==pilier])
    n25 = len(obj_25[obj_25['pilier_court']==pilier])
    ae24p = obj_24[obj_24['pilier_court']==pilier]['ae'].sum()/1e9
    ae25p = obj_25[obj_25['pilier_court']==pilier]['ae'].sum()/1e9
    print(f'   {pilier:30s}: N {n24}→{n25} obj. | AE {ae24p:.2f}→{ae25p:.2f} Mds')
print(f'   Chi² piliers : χ²={chi2_p:.2f}, p={p_chi2_p:.4f} {stars(p_chi2_p)}')
print(f'   Programmes stables (même pilier) : {n_stable}/{n_commun} ({n_stable/n_commun*100:.1f}%)')

print('\n📌 5. CLUSTERING')
km_24_vc = art_cl_24['cluster_kmeans'].value_counts().sort_index()
km_25_vc = art_cl_25['cluster_kmeans'].value_counts().sort_index()
print(f'   K-Means 3 clusters | 2024 : {dict(km_24_vc)} | 2025 : {dict(km_25_vc)}')
noise_24 = (art_cl_24['cluster_hdbscan']==-1).sum()
noise_25 = (art_cl_25['cluster_hdbscan']==-1).sum()
print(f'   HDBSCAN bruit | 2024 : {noise_24}/{len(art_cl_24)} ({noise_24/len(art_cl_24)*100:.1f}%) | 2025 : {noise_25}/{len(art_cl_25)} ({noise_25/len(art_cl_25)*100:.1f}%)')
print(f'   Kruskal-Wallis (sim. vs clusters) : H={kw_stat:.4f}, p={kw_p:.4f} {stars(kw_p)}')

print('\n' + '='*80)
sig_count = sum(1 for _,p in all_tests if p<0.05)
print(f'  ✅ {sig_count}/{len(all_tests)} tests significatifs au seuil α=5%')
print('='*80)

### 7.2 Conclusions et Recommandations

#### 🔑 Principaux résultats

1. **Glissement sémantique modéré** : La loi 2025 présente une similarité cosinus moyenne plus élevée que 2024,
   suggérant une plus grande continuité textuelle dans les articles de 2025 comparés à leurs analogues 2024.
   Le KS-test confirme que les deux distributions sont **significativement différentes**.

2. **Rééquilibrage budgétaire marqué** : Le pilier *Transformation structurelle* connaît la plus forte 
   croissance (+143%), tandis que *Capital humain* recule (−6.5%). Ce rééquilibrage est statistiquement 
   significatif (Chi² des piliers SND30).

3. **Concentration budgétaire stable** : L'indice de Gini et le HHI entre piliers restent similaires 
   entre 2024 et 2025, indiquant une structure de concentration stable.

4. **Stabilité thématique (LDA)** : Le Chi² sur les topics dominants indique si la distribution 
   thématique des articles a significativement évolué.

5. **Cohérence SND30** : Les programmes restent majoritairement classifiés dans le même pilier 
   entre les deux années (stabilité de la matrice de transition).

#### 📋 Recommandations

- Approfondir l'analyse des articles sous le seuil de similarité θ=0.70 (changements les plus importants)
- Croiser les scores de classification SND30 avec les indicateurs de performance budgétaire (taux d'exécution)
- Étendre l'analyse à l'exécution budgétaire pour mesurer l'alignement programmation/réalisation
- Appliquer une correction de Bonferroni pour les tests multiples (tests sur 4 piliers)


In [ ]:
# ─── Export du rapport de synthèse ────────────────────────────────────────────
print('\n📁 Fichiers générés dans outputs/ :')
import os
output_files = [f for f in os.listdir('/mnt/user-data/outputs/') if f.endswith('.png')]
for f in sorted(output_files):
    print(f'  ✓ {f}')
print()
print('✅ Analyse complète terminée !')